In [1]:
# ============================================================
# PHASE 3 — CELL 1
# Environment Check
# ============================================================

import sys
import torch
import onnxruntime as ort

print("=" * 70)
print("PHASE 3 — ENVIRONMENT CHECK")
print("=" * 70)

print("Python :", sys.executable)
print("Torch  :", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count :", torch.cuda.device_count())
    print("GPU      :", torch.cuda.get_device_name(0))
else:
    print("GPU      : NOT AVAILABLE")

print()
print("ONNX Runtime :", ort.__version__)
print("Providers    :", ort.get_available_providers())

print("=" * 70)

PHASE 3 — ENVIRONMENT CHECK
Python : c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe
Torch  : 2.13.0+cu126
CUDA   : 12.6
CUDA available : True
GPU count : 1
GPU      : NVIDIA GeForce RTX 4050 Laptop GPU

ONNX Runtime : 1.27.0
Providers    : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [2]:
# ============================================================
# PHASE 3 — CELL 2
# Imports
# ============================================================

import os
import cv2
import json
import gc
import shutil
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime, timezone

from tqdm.auto import tqdm

from insightface.app import FaceAnalysis
from skimage.transform import SimilarityTransform

In [3]:
# ============================================================
# PHASE 3 — CELL 3
# Production Configuration
# ============================================================

PROJECT_DIR = Path(
    r"C:\LipReadingSSL"
).resolve()

OUTPUT_DIR = (
    PROJECT_DIR / "output"
)

SUPPORTED_VIDEO_EXTENSIONS = {
    ".webm",
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".mpg",
    ".mpeg",
}

IMAGE_SIZE = 256

GPU_ID = 0

DET_SIZE = (
    640,
    640
)

MIN_FACE_SCORE = 0.50

FORCE_RERUN = False

SAVE_ALIGNMENT_PREVIEW = True

SAVE_ORIGINAL_LANDMARK = True

SAVE_ALIGNED_LANDMARK = True

SAVE_TRANSFORM = True

SAVE_METADATA = True

# ถ้า True จะข้าม frame ที่ Phase 2 บอกว่าไม่มีหน้า
USE_FACE_DETECTED_FILTER = True

print("=" * 70)
print("PHASE 3 — PRODUCTION CONFIGURATION")
print("=" * 70)

print("Project       :", PROJECT_DIR)
print("Output        :", OUTPUT_DIR)
print("Image Size    :", IMAGE_SIZE)
print("GPU ID        :", GPU_ID)
print("Detection     :", MIN_FACE_SCORE)
print("Force Rerun   :", FORCE_RERUN)

print("=" * 70)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"Output directory not found:\n{OUTPUT_DIR}"
    )

PHASE 3 — PRODUCTION CONFIGURATION
Project       : C:\LipReadingSSL
Output        : C:\LipReadingSSL\output
Image Size    : 256
GPU ID        : 0
Detection     : 0.5
Force Rerun   : False


In [4]:
# ============================================================
# PHASE 3 — CELL 4
# Unified Production Path Helpers
# ============================================================

def utc_now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()


# ============================================================
# Video ID
# ============================================================

def get_video_id(video_path):

    return Path(
        video_path
    ).stem


# ============================================================
# Video output directory
# ============================================================

def get_video_dir(video_path):

    video_id = get_video_id(
        video_path
    )

    return (
        OUTPUT_ROOT /
        video_id
    )


# ============================================================
# Unified Phase 3 paths
# ============================================================

def get_phase3_paths(video_path):

    video_path = Path(
        video_path
    ).resolve()

    video_id = get_video_id(
        video_path
    )

    output_dir = (
        OUTPUT_ROOT /
        video_id
    )

    paths = {

        # ----------------------------------------------------
        # Root
        # ----------------------------------------------------

        "output_dir":
            output_dir,

        # ----------------------------------------------------
        # Phase 2 input
        # ----------------------------------------------------

        "detection_csv":
            output_dir /
            DETECTION_CSV_NAME,

        # ----------------------------------------------------
        # Phase 3 output
        # ----------------------------------------------------

        "aligned_face":
            output_dir /
            ALIGNED_FACE_DIR_NAME,

        "landmarks":
            output_dir /
            LANDMARK_DIR_NAME,

        "alignment_metadata":
            output_dir /
            ALIGNMENT_METADATA_NAME,

        # ----------------------------------------------------
        # Phase 3 state
        # ----------------------------------------------------

        "phase3_state":
            output_dir /
            "phase3_state.json",

        # ----------------------------------------------------
        # Skipped frames
        # ----------------------------------------------------

        "skipped":
            output_dir /
            "phase3_skipped_frames.csv",

        # ----------------------------------------------------
        # Backup
        # ----------------------------------------------------

        "backup_dir":
            output_dir /
            "backups",
    }

    return paths


# ============================================================
# JSON helpers
# ============================================================

def read_json(path):

    path = Path(path)

    if not path.exists():
        return {}

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    except Exception:

        return {}


def write_json(path, data):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp = path.with_name(
        path.name + ".tmp"
    )

    with open(
        temp,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )

    temp.replace(path)


print("=" * 70)
print("PHASE 3 — UNIFIED PATH HELPERS READY")
print("=" * 70)

PHASE 3 — UNIFIED PATH HELPERS READY


In [5]:
# ============================================================
# PHASE 3 — CELL 5
# Production Output Structure
# ============================================================

def create_output_structure(video_path):

    video_path = Path(
        video_path
    ).resolve()

    if not video_path.exists():
        raise FileNotFoundError(
            f"Video not found:\n{video_path}"
        )

    if not video_path.is_file():
        raise RuntimeError(
            f"Video path is not a file:\n{video_path}"
        )

    paths = get_phase3_paths(
        video_path
    )

    directories = [
        paths["root"],
        paths["aligned_face"],
        paths["landmarks"],
        paths["backup_dir"],
    ]

    for directory in directories:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    return paths


print("=" * 70)
print("PHASE 3 — OUTPUT STRUCTURE READY")
print("=" * 70)

PHASE 3 — OUTPUT STRUCTURE READY


In [6]:
# ============================================================
# PHASE 3 — CELL 6
# Load Phase 2 Detection CSV
# ============================================================

REQUIRED_COLUMNS = [
    "video_id",
    "frame_index",
    "timestamp",
    "face_detected",
]


def load_detection_csv(
    video_path
):

    paths = get_phase3_paths(
        video_path
    )

    csv_path = paths[
        "detection"
    ]

    if not csv_path.exists():

        raise FileNotFoundError(
            f"Detection CSV not found:\n"
            f"{csv_path}"
        )

    df = pd.read_csv(
        csv_path
    )

    missing = [

        col

        for col in REQUIRED_COLUMNS

        if col not in df.columns

    ]

    if missing:

        raise ValueError(
            "Detection CSV missing columns: "
            + ", ".join(missing)
        )

    # --------------------------------------------------------
    # Normalize frame index
    # --------------------------------------------------------

    df["frame_index"] = pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )

    df = df[
        df["frame_index"].notna()
    ].copy()

    df["frame_index"] = (
        df["frame_index"]
        .astype(int)
    )

    # --------------------------------------------------------
    # Normalize face_detected
    # --------------------------------------------------------

    df["face_detected"] = (
        df["face_detected"]
        .astype(str)
        .str.lower()
        .isin([
            "true",
            "1",
            "yes"
        ])
    )

    df = df.sort_values(
        "frame_index"
    )

    df = df.drop_duplicates(
        subset=["frame_index"],
        keep="last"
    )

    return df


def get_target_frames(
    video_path
):

    df = load_detection_csv(
        video_path
    )

    if USE_FACE_DETECTED_FILTER:

        target = df[
            df["face_detected"] == True
        ].copy()

    else:

        target = df.copy()

    return target


print(
    "Detection CSV loader ready."
)

Detection CSV loader ready.


In [7]:
# ============================================================
# PHASE 3 — CELL 7
# InsightFace Initialization
# ============================================================

providers = [
    "CUDAExecutionProvider",
    "CPUExecutionProvider"
]

if (
    "CUDAExecutionProvider"
    not in ort.get_available_providers()
):

    print(
        "WARNING: CUDAExecutionProvider "
        "not available."
    )

    providers = [
        "CPUExecutionProvider"
    ]


app = FaceAnalysis(
    name="buffalo_l",
    providers=providers
)

ctx_id = (
    GPU_ID
    if "CUDAExecutionProvider"
    in providers
    else -1
)

app.prepare(
    ctx_id=ctx_id,
    det_size=DET_SIZE
)

print("=" * 70)
print("InsightFace Ready")
print("Providers :", providers)
print("Context   :", ctx_id)
print("=" * 70)

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127

In [8]:
# ============================================================
# PHASE 3 — CELL 8
# Face Alignment
# ============================================================

# ArcFace 5-point template
SRC = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041],
], dtype=np.float32)


def estimate_transform(
    kps,
    image_size=256
):

    src = SRC.copy()

    src *= (
        image_size / 112.0
    )

    tform = SimilarityTransform()

    success = tform.estimate(
        kps.astype(np.float32),
        src
    )

    if not success:

        raise RuntimeError(
            "SimilarityTransform estimation failed."
        )

    return tform.params[:2].astype(
        np.float32
    )


def align_face(
    image,
    face
):

    if face.kps is None:

        raise RuntimeError(
            "Face 5-point landmarks unavailable."
        )

    M = estimate_transform(
        face.kps,
        IMAGE_SIZE
    )

    aligned = cv2.warpAffine(
        image,
        M,
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    return aligned, M


def transform_landmarks(
    landmarks,
    M
):

    landmarks = np.asarray(
        landmarks,
        dtype=np.float32
    )

    ones = np.ones(
        (
            landmarks.shape[0],
            1
        ),
        dtype=np.float32
    )

    pts = np.concatenate(
        [
            landmarks,
            ones
        ],
        axis=1
    )

    aligned = (
        M @ pts.T
    ).T

    return aligned.astype(
        np.float32
    )


print(
    "Alignment functions ready."
)

Alignment functions ready.


In [9]:
# ============================================================
# PHASE 3 — CELL 9
# Optional Preview
# ============================================================

def draw_landmarks(
    image,
    landmarks,
    radius=1
):

    canvas = image.copy()

    for x, y in landmarks:

        cv2.circle(
            canvas,
            (
                int(round(x)),
                int(round(y))
            ),
            radius,
            (0, 255, 0),
            -1
        )

    return canvas


def save_preview(
    paths,
    original,
    aligned,
    landmark_original,
    landmark_aligned
):

    preview_dir = (
        paths["root"] /
        "alignment_preview"
    )

    preview_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    original_drawn = draw_landmarks(
        original,
        landmark_original
    )

    aligned_drawn = draw_landmarks(
        aligned,
        landmark_aligned
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 5)
    )

    axes[0].imshow(
        cv2.cvtColor(
            original_drawn,
            cv2.COLOR_BGR2RGB
        )
    )

    axes[0].set_title(
        "Original"
    )

    axes[0].axis("off")

    axes[1].imshow(
        cv2.cvtColor(
            aligned_drawn,
            cv2.COLOR_BGR2RGB
        )
    )

    axes[1].set_title(
        "Aligned"
    )

    axes[1].axis("off")

    plt.tight_layout()

    fig.savefig(
        preview_dir /
        "preview.jpg",
        dpi=200
    )

    plt.close(fig)

In [19]:
# ============================================================
# PHASE 3 — CELL 10
# FAST PRODUCTION CONFIGURATION + PATHS
# ============================================================

from pathlib import Path
import pandas as pd
import cv2
import numpy as np
import json
import traceback
import gc
import time


# ============================================================
# Project
# ============================================================

PROJECT_ROOT = Path(
    r"C:\LipReadingSSL"
).resolve()

RAW_VIDEO_DIR = (
    PROJECT_ROOT /
    "dataset" /
    "raw_videos"
)

OUTPUT_ROOT = (
    PROJECT_ROOT /
    "output"
)


# ============================================================
# Supported videos
# ============================================================

SUPPORTED_VIDEO_EXTENSIONS = {
    ".webm",
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".mpg",
    ".mpeg",
}


# ============================================================
# Phase 2 input
# ============================================================

DETECTION_CSV_NAME = "detection.csv"


# ============================================================
# Phase 3 output
# ============================================================

ALIGNED_FACE_DIR_NAME = "aligned_face"

LANDMARK_DIR_NAME = "landmarks"

ALIGNMENT_METADATA_NAME = (
    "alignment_metadata.csv"
)

SKIPPED_NAME = "skipped.csv"

PHASE3_STATE_NAME = (
    "phase3_state.json"
)


# ============================================================
# Image
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}


# ============================================================
# Processing settings
# ============================================================

IMAGE_SIZE = 224

MIN_FACE_SCORE = 0.40

USE_FACE_DETECTED_FILTER = True

SAVE_METADATA = True

SAVE_TRANSFORM = True


# ============================================================
# Resume / rerun
# ============================================================

FORCE_RERUN = False


# ============================================================
# FAST settings
# ============================================================

# แสดง progress ทุกกี่ frames
PROGRESS_EVERY = 500


# ถ้า target frame ทั้งหมดผ่านแล้ว ให้หยุดอ่านวิดีโอทันที
STOP_AFTER_LAST_TARGET = True


# ใช้ max_num=1 กับ InsightFace
# เพราะ Phase 3 ต้องการ face หลักเพียงตัวเดียว
USE_MAX_ONE_FACE = True


# ============================================================
# Output root
# ============================================================

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("PHASE 3 — FAST PRODUCTION CONFIGURATION")
print("=" * 70)

print(
    f"Project root : {PROJECT_ROOT}"
)

print(
    f"Raw videos   : {RAW_VIDEO_DIR}"
)

print(
    f"Output root  : {OUTPUT_ROOT}"
)

print(
    f"Image size   : {IMAGE_SIZE}"
)

print(
    f"Min face score: {MIN_FACE_SCORE}"
)

print(
    f"Filter face  : {USE_FACE_DETECTED_FILTER}"
)

print(
    f"Progress every: {PROGRESS_EVERY}"
)

print(
    f"Force rerun  : {FORCE_RERUN}"
)

print("=" * 70)

PHASE 3 — FAST PRODUCTION CONFIGURATION
Project root : C:\LipReadingSSL
Raw videos   : C:\LipReadingSSL\dataset\raw_videos
Output root  : C:\LipReadingSSL\output
Image size   : 224
Min face score: 0.4
Filter face  : True
Progress every: 500
Force rerun  : False


In [20]:
# ============================================================
# PHASE 3 — CELL 11
# FAST PATH MANAGEMENT + PHASE 2 VALIDATION
# ============================================================


# ============================================================
# Get Phase 3 paths
# ============================================================

def get_phase3_paths(video_path):

    video_path = Path(
        video_path
    ).resolve()

    video_id = get_video_id(
        video_path
    )

    output_dir = (
        OUTPUT_ROOT /
        video_id
    )

    return {

        # ----------------------------------------------------
        # Input
        # ----------------------------------------------------

        "video": video_path,

        "detection_csv": (
            output_dir /
            DETECTION_CSV_NAME
        ),

        # ----------------------------------------------------
        # Output
        # ----------------------------------------------------

        "output_dir": output_dir,

        "aligned_face": (
            output_dir /
            ALIGNED_FACE_DIR_NAME
        ),

        "landmarks": (
            output_dir /
            LANDMARK_DIR_NAME
        ),

        "alignment_metadata": (
            output_dir /
            ALIGNMENT_METADATA_NAME
        ),

        "skipped": (
            output_dir /
            SKIPPED_NAME
        ),

        "phase3_state": (
            output_dir /
            PHASE3_STATE_NAME
        ),

        # ----------------------------------------------------
        # Backup
        # ----------------------------------------------------

        "backup_dir": (
            output_dir /
            "backup"
        ),
    }


# ============================================================
# Create output structure
# ============================================================

def create_phase3_output_structure(
    video_path
):

    video_path = Path(
        video_path
    ).resolve()

    if not video_path.exists():

        raise FileNotFoundError(
            f"Video not found:\n"
            f"{video_path}"
        )

    if not video_path.is_file():

        raise RuntimeError(
            f"Video path is not a file:\n"
            f"{video_path}"
        )

    paths = get_phase3_paths(
        video_path
    )

    paths["output_dir"].mkdir(
        parents=True,
        exist_ok=True
    )

    paths["aligned_face"].mkdir(
        parents=True,
        exist_ok=True
    )

    paths["landmarks"].mkdir(
        parents=True,
        exist_ok=True
    )

    paths["backup_dir"].mkdir(
        parents=True,
        exist_ok=True
    )

    return paths


# ============================================================
# Required Phase 2 columns
# ============================================================

REQUIRED_PHASE2_COLUMNS = [
    "video_id",
    "frame_index",
    "timestamp",
    "face_detected",
]


# ============================================================
# Validate Phase 2 input
# ============================================================

def validate_phase2_input(
    video_path
):

    video_path = Path(
        video_path
    ).resolve()

    video_id = get_video_id(
        video_path
    )

    paths = get_phase3_paths(
        video_path
    )

    csv_path = paths[
        "detection_csv"
    ]

    result = {

        "valid": False,

        "video_id": video_id,

        "csv": str(csv_path),

        "rows": 0,

        "reason": None,

        "missing_columns": [],
    }


    # ========================================================
    # Video
    # ========================================================

    if not video_path.exists():

        result["reason"] = (
            "missing_video"
        )

        return result


    # ========================================================
    # Detection CSV
    # ========================================================

    if not csv_path.exists():

        result["reason"] = (
            "missing_detection_csv"
        )

        return result


    # ========================================================
    # Read CSV
    # ========================================================

    try:

        df = pd.read_csv(
            csv_path
        )

    except Exception as e:

        result["reason"] = (
            "csv_read_error:"
            f"{type(e).__name__}"
        )

        return result


    result["rows"] = len(df)


    # ========================================================
    # Columns
    # ========================================================

    missing = [

        col

        for col in REQUIRED_PHASE2_COLUMNS

        if col not in df.columns

    ]

    result["missing_columns"] = (
        missing
    )

    if missing:

        result["reason"] = (
            "missing_columns:"
            +
            ",".join(missing)
        )

        return result


    # ========================================================
    # Empty
    # ========================================================

    if df.empty:

        result["reason"] = (
            "empty_detection_csv"
        )

        return result


    # ========================================================
    # Video ID
    # ========================================================

    video_ids = set(

        df["video_id"]
        .dropna()
        .astype(str)

    )

    if video_ids:

        if video_ids != {video_id}:

            result["reason"] = (
                "video_id_mismatch"
            )

            return result


    # ========================================================
    # Frame index
    # ========================================================

    frame_index = pd.to_numeric(

        df["frame_index"],

        errors="coerce"

    )

    if frame_index.isna().any():

        result["reason"] = (
            "invalid_frame_index"
        )

        return result


    result["valid"] = True

    result["reason"] = (
        "valid_phase2_input"
    )

    return result


print("=" * 70)
print(
    "PHASE 3 — FAST PATH MANAGEMENT + VALIDATION READY"
)
print("=" * 70)

PHASE 3 — FAST PATH MANAGEMENT + VALIDATION READY


In [12]:
import inspect

source = inspect.getsource(process_video)

lines = source.splitlines()

for i, line in enumerate(lines, start=1):
    if "paths" in line:
        print(f"{i:03d}: {line}")

NameError: name 'process_video' is not defined

In [21]:
# ============================================================
# PHASE 3 — CELL 12
# FAST PRODUCTION FACE ALIGNMENT PROCESSOR
# ============================================================

def process_video(
    video_path
):

    start_time = time.time()

    video_path = Path(
        video_path
    ).resolve()

    video_id = get_video_id(
        video_path
    )


    # ========================================================
    # Output structure
    # ========================================================

    paths = create_phase3_output_structure(
        video_path
    )


    print()
    print("=" * 70)
    print(
        f"PHASE 3 FAST PROCESSING : {video_id}"
    )
    print("=" * 70)

    print(
        f"Input     : {video_path}"
    )

    print(
        f"Output    : {paths['output_dir']}"
    )

    print(
        f"Aligned   : {paths['aligned_face']}"
    )

    print(
        f"Landmarks : {paths['landmarks']}"
    )

    print(
        f"Metadata  : {paths['alignment_metadata']}"
    )


    # ========================================================
    # Validate Phase 2
    # ========================================================

    validation = validate_phase2_input(
        video_path
    )

    if not validation["valid"]:

        raise RuntimeError(
            "Phase 2 input invalid: "
            +
            str(
                validation["reason"]
            )
        )


    # ========================================================
    # Load detection CSV
    # ========================================================

    detection_df = pd.read_csv(
        paths["detection_csv"]
    )


    # ========================================================
    # Normalize frame index
    # ========================================================

    detection_df["frame_index"] = (
        pd.to_numeric(
            detection_df["frame_index"],
            errors="coerce"
        )
    )

    detection_df = detection_df[
        detection_df["frame_index"].notna()
    ].copy()

    detection_df["frame_index"] = (
        detection_df["frame_index"]
        .astype(np.int64)
    )


    # ========================================================
    # Normalize face_detected
    # ========================================================

    detection_df["face_detected"] = (

        detection_df["face_detected"]
        .astype(str)
        .str.lower()
        .isin([
            "true",
            "1",
            "yes"
        ])

    )


    # ========================================================
    # Remove duplicate frames
    # ========================================================

    detection_df = (

        detection_df

        .drop_duplicates(
            subset=[
                "frame_index"
            ],
            keep="last"
        )

        .sort_values(
            "frame_index"
        )

        .reset_index(
            drop=True
        )

    )


    # ========================================================
    # Target frames
    # ========================================================

    if USE_FACE_DETECTED_FILTER:

        target_df = detection_df[
            detection_df[
                "face_detected"
            ]
        ].copy()

    else:

        target_df = detection_df.copy()


    target_df = (

        target_df

        .sort_values(
            "frame_index"
        )

        .reset_index(
            drop=True
        )

    )


    # ========================================================
    # FAST LOOKUP
    # ========================================================

    target_frames = set(
        target_df[
            "frame_index"
        ].tolist()
    )


    target_info = (

        target_df

        .set_index(
            "frame_index"
        )

        .to_dict(
            orient="index"
        )

    )


    total_target = len(
        target_frames
    )


    print()
    print(
        f"Phase 2 rows : {len(detection_df)}"
    )

    print(
        f"Target frames: {total_target}"
    )


    if total_target == 0:

        print(
            "No target frames."
        )

        return {

            "video_id":
                video_id,

            "status":
                "COMPLETED",

            "phase2_rows":
                len(detection_df),

            "target_frames":
                0,

            "aligned_frames":
                0,

            "skipped_frames":
                0,

            "failed_frames":
                0,

        }


    # ========================================================
    # Existing output lookup
    #
    # สำคัญมาก:
    # ถ้า output มีอยู่แล้ว จะไม่เรียก InsightFace ซ้ำ
    # ========================================================

    existing_aligned = {
        p.stem
        for p in paths[
            "aligned_face"
        ].iterdir()
        if (
            p.is_file()
            and
            p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    }


    existing_landmarks = {
        p.stem
        for p in paths[
            "landmarks"
        ].iterdir()
        if (
            p.is_file()
            and
            p.suffix.lower()
            == ".npz"
        )
    }


    # ========================================================
    # Existing metadata
    # ========================================================

    metadata_path = paths[
        "alignment_metadata"
    ]

    if metadata_path.exists():

        try:

            old_metadata_df = pd.read_csv(
                metadata_path
            )

        except Exception:

            old_metadata_df = pd.DataFrame()

    else:

        old_metadata_df = pd.DataFrame()


    existing_metadata_frames = set()

    if (
        not old_metadata_df.empty
        and
        "frame_index"
        in old_metadata_df.columns
    ):

        existing_metadata_frames = set(

            pd.to_numeric(
                old_metadata_df[
                    "frame_index"
                ],
                errors="coerce"
            )
            .dropna()
            .astype(int)
            .tolist()

        )


    # ========================================================
    # Determine frames that REALLY need processing
    # ========================================================

    frames_to_process = []

    frames_already_done = 0

    for frame_index in target_frames:

        frame_name = (
            f"frame_{frame_index:06d}"
        )

        aligned_exists = (
            frame_name
            in existing_aligned
        )

        landmark_exists = (
            frame_name
            in existing_landmarks
        )

        if (
            not FORCE_RERUN
            and aligned_exists
            and landmark_exists
        ):

            frames_already_done += 1

        else:

            frames_to_process.append(
                frame_index
            )


    frames_to_process.sort()


    print()
    print(
        f"Already done : "
        f"{frames_already_done}"
    )

    print(
        f"Need process : "
        f"{len(frames_to_process)}"
    )


    # ========================================================
    # Nothing to process
    # ========================================================

    if not frames_to_process:

        print()
        print(
            "All target frames already processed."
        )

        # ----------------------------------------------------
        # Ensure metadata contains all target frames
        # ----------------------------------------------------

        if SAVE_METADATA:

            missing_metadata = (
                target_frames
                -
                existing_metadata_frames
            )

            if missing_metadata:

                repair_records = []

                for frame_index in sorted(
                    missing_metadata
                ):

                    row = target_info[
                        frame_index
                    ]

                    frame_name = (
                        f"frame_{frame_index:06d}"
                    )

                    aligned_path = (
                        paths["aligned_face"]
                        /
                        f"{frame_name}.jpg"
                    )

                    landmark_path = (
                        paths["landmarks"]
                        /
                        f"{frame_name}.npz"
                    )

                    repair_records.append({

                        "video_id":
                            video_id,

                        "frame_index":
                            frame_index,

                        "timestamp":
                            row.get(
                                "timestamp"
                            ),

                        "track_id":
                            row.get(
                                "track_id"
                            ),

                        "det_score":
                            np.nan,

                        "original_width":
                            np.nan,

                        "original_height":
                            np.nan,

                        "aligned_width":
                            IMAGE_SIZE,

                        "aligned_height":
                            IMAGE_SIZE,

                        "aligned_face":
                            str(
                                aligned_path
                            ),

                        "landmarks":
                            str(
                                landmark_path
                            ),

                        "status":
                            "aligned_existing",

                        "processed_at":
                            utc_now_iso(),

                    })

                if repair_records:

                    repair_df = pd.DataFrame(
                        repair_records
                    )

                    old_metadata_df = pd.concat(
                        [
                            old_metadata_df,
                            repair_df
                        ],
                        ignore_index=True
                    )


                    old_metadata_df = (

                        old_metadata_df

                        .drop_duplicates(
                            subset=[
                                "video_id",
                                "frame_index"
                            ],
                            keep="last"
                        )

                        .sort_values(
                            "frame_index"
                        )

                    )

                    old_metadata_df.to_csv(
                        metadata_path,
                        index=False,
                        encoding="utf-8-sig"
                    )


        elapsed = (
            time.time()
            -
            start_time
        )

        return {

            "video_id":
                video_id,

            "status":
                "COMPLETED",

            "phase2_rows":
                len(detection_df),

            "target_frames":
                total_target,

            "aligned_frames":
                total_target,

            "skipped_frames":
                0,

            "failed_frames":
                0,

            "elapsed_seconds":
                elapsed,

        }


    # ========================================================
    # Open video ONCE
    # ========================================================

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open video:\n"
            f"{video_path}"
        )


    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    print()
    print(
        f"FPS         : {fps}"
    )

    print(
        f"Video frames: {frame_count}"
    )


    # ========================================================
    # FAST target bounds
    # ========================================================

    first_target = (
        frames_to_process[0]
    )

    last_target = (
        frames_to_process[-1]
    )

    target_process_set = set(
        frames_to_process
    )


    # ========================================================
    # Statistics
    # ========================================================

    processed = 0

    skipped = 0

    failed = 0

    metadata_records = []

    skipped_records = []


    # ========================================================
    # Single timestamp for batch
    #
    # ลดการเรียก utc_now_iso() หลายหมื่นครั้ง
    # ========================================================

    processing_timestamp = (
        utc_now_iso()
    )


    # ========================================================
    # Sequential read
    # ========================================================

    current_frame = 0


    try:

        while True:

            success, frame = cap.read()

            if not success:

                break


            # ------------------------------------------------
            # Stop after last required frame
            # ------------------------------------------------

            if (
                STOP_AFTER_LAST_TARGET
                and
                current_frame >
                last_target
            ):

                break


            # ------------------------------------------------
            # Skip frames before first target
            # ------------------------------------------------

            if (
                current_frame
                not in target_process_set
            ):

                current_frame += 1

                continue


            row = target_info[
                current_frame
            ]


            timestamp = row.get(
                "timestamp"
            )

            track_id = row.get(
                "track_id"
            )


            # =================================================
            # InsightFace
            # =================================================

            try:

                if USE_MAX_ONE_FACE:

                    try:

                        faces = app.get(
                            frame,
                            max_num=1
                        )

                    except TypeError:

                        faces = app.get(
                            frame
                        )

                else:

                    faces = app.get(
                        frame
                    )

            except Exception as e:

                failed += 1

                skipped_records.append({

                    "video_id":
                        video_id,

                    "frame_index":
                        current_frame,

                    "reason":
                        (
                            "insightface_error:"
                            f"{type(e).__name__}"
                        ),

                    "timestamp":
                        timestamp,

                })

                current_frame += 1

                continue


            # =================================================
            # No face
            # =================================================

            if not faces:

                skipped += 1

                skipped_records.append({

                    "video_id":
                        video_id,

                    "frame_index":
                        current_frame,

                    "reason":
                        "no_face_found",

                    "timestamp":
                        timestamp,

                })

                current_frame += 1

                continue


            # =================================================
            # Best face
            # =================================================

            if len(faces) > 1:

                face = max(

                    faces,

                    key=lambda f:
                        float(
                            getattr(
                                f,
                                "det_score",
                                0.0
                            )
                        )

                )

            else:

                face = faces[0]


            det_score = float(
                getattr(
                    face,
                    "det_score",
                    0.0
                )
            )


            # =================================================
            # Score
            # =================================================

            if (
                det_score
                <
                MIN_FACE_SCORE
            ):

                skipped += 1

                skipped_records.append({

                    "video_id":
                        video_id,

                    "frame_index":
                        current_frame,

                    "reason":
                        "face_score_below_threshold",

                    "timestamp":
                        timestamp,

                    "det_score":
                        det_score,

                })

                current_frame += 1

                continue


            # =================================================
            # 5-point landmarks
            # =================================================

            if face.kps is None:

                skipped += 1

                skipped_records.append({

                    "video_id":
                        video_id,

                    "frame_index":
                        current_frame,

                    "reason":
                        "missing_5point_landmarks",

                    "timestamp":
                        timestamp,

                })

                current_frame += 1

                continue


            # =================================================
            # Alignment
            # =================================================

            try:

                aligned_face, M = (
                    align_face(
                        frame,
                        face
                    )
                )

            except Exception as e:

                failed += 1

                skipped_records.append({

                    "video_id":
                        video_id,

                    "frame_index":
                        current_frame,

                    "reason":
                        (
                            "alignment_error:"
                            f"{type(e).__name__}"
                        ),

                    "timestamp":
                        timestamp,

                })

                current_frame += 1

                continue


            # =================================================
            # Landmarks
            # =================================================

            original_kps = np.asarray(
                face.kps,
                dtype=np.float32
            )

            aligned_kps = (
                transform_landmarks(
                    original_kps,
                    M
                )
            )


            # =================================================
            # Paths
            # =================================================

            frame_name = (
                f"frame_{current_frame:06d}"
            )

            aligned_path = (
                paths["aligned_face"]
                /
                f"{frame_name}.jpg"
            )

            landmark_path = (
                paths["landmarks"]
                /
                f"{frame_name}.npz"
            )


            # =================================================
            # Save aligned face
            # =================================================

            if (
                FORCE_RERUN
                or
                not aligned_path.exists()
            ):

                ok = cv2.imwrite(
                    str(aligned_path),
                    aligned_face
                )

                if not ok:

                    failed += 1

                    skipped_records.append({

                        "video_id":
                            video_id,

                        "frame_index":
                            current_frame,

                        "reason":
                            "aligned_face_write_failed",

                        "timestamp":
                            timestamp,

                    })

                    current_frame += 1

                    continue


            # =================================================
            # Save landmarks
            # =================================================

            if (
                FORCE_RERUN
                or
                not landmark_path.exists()
            ):

                save_data = {

                    "original_kps":
                        original_kps,

                    "aligned_kps":
                        aligned_kps,

                }

                if SAVE_TRANSFORM:

                    save_data[
                        "transform"
                    ] = M


                np.savez(
                    str(landmark_path),
                    **save_data
                )


            # =================================================
            # Metadata
            # =================================================

            metadata_records.append({

                "video_id":
                    video_id,

                "frame_index":
                    current_frame,

                "timestamp":
                    timestamp,

                "track_id":
                    track_id,

                "det_score":
                    det_score,

                "original_width":
                    int(
                        frame.shape[1]
                    ),

                "original_height":
                    int(
                        frame.shape[0]
                    ),

                "aligned_width":
                    IMAGE_SIZE,

                "aligned_height":
                    IMAGE_SIZE,

                "aligned_face":
                    str(
                        aligned_path
                    ),

                "landmarks":
                    str(
                        landmark_path
                    ),

                "status":
                    "aligned",

                "processed_at":
                    processing_timestamp,

            })


            processed += 1


            # =================================================
            # Progress
            # =================================================

            if (
                processed % PROGRESS_EVERY == 0
                or
                processed ==
                len(frames_to_process)
            ):

                elapsed = (
                    time.time()
                    -
                    start_time
                )

                speed = (
                    processed /
                    elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"Progress: "
                    f"{processed}/"
                    f"{len(frames_to_process)} "
                    f"| "
                    f"frame={current_frame} "
                    f"| "
                    f"{speed:.2f} proc/s"
                )


            current_frame += 1


    finally:

        cap.release()


    # ========================================================
    # Save metadata
    # ========================================================

    if SAVE_METADATA:

        new_metadata_df = pd.DataFrame(
            metadata_records
        )

        if not new_metadata_df.empty:

            if (
                not old_metadata_df.empty
                and
                not FORCE_RERUN
            ):

                combined_df = pd.concat(
                    [
                        old_metadata_df,
                        new_metadata_df
                    ],
                    ignore_index=True
                )

            else:

                combined_df = (
                    new_metadata_df
                )


            combined_df = (

                combined_df

                .drop_duplicates(
                    subset=[
                        "video_id",
                        "frame_index"
                    ],
                    keep="last"
                )

                .sort_values(
                    "frame_index"
                )

            )


            combined_df.to_csv(
                metadata_path,
                index=False,
                encoding="utf-8-sig"
            )


    # ========================================================
    # Save skipped
    # ========================================================

    skipped_path = paths[
        "skipped"
    ]

    if skipped_records:

        skipped_df = pd.DataFrame(
            skipped_records
        )

        skipped_df.to_csv(
            skipped_path,
            index=False,
            encoding="utf-8-sig"
        )


    # ========================================================
    # Save state
    # ========================================================

    elapsed = (
        time.time()
        -
        start_time
    )


    state = {

        "video_id":
            video_id,

        "video_path":
            str(video_path),

        "status":
            "COMPLETED",

        "fps":
            float(fps)
            if fps
            else None,

        "frame_count":
            frame_count,

        "phase2_rows":
            int(len(detection_df)),

        "target_frames":
            int(total_target),

        "aligned_frames":
            int(
                frames_already_done
                +
                processed
            ),

        "skipped_frames":
            int(skipped),

        "failed_frames":
            int(failed),

        "image_size":
            IMAGE_SIZE,

        "min_face_score":
            MIN_FACE_SCORE,

        "elapsed_seconds":
            float(elapsed),

        "processing_speed":
            float(
                processed / elapsed
            )
            if elapsed > 0
            else 0,

        "completed_at":
            utc_now_iso(),

    }


    write_json(
        paths[
            "phase3_state"
        ],
        state
    )


    # ========================================================
    # Statistics
    # ========================================================

    total_aligned = (
        frames_already_done
        +
        processed
    )


    stats = {

        "video_id":
            video_id,

        "status":
            "COMPLETED",

        "phase2_rows":
            len(detection_df),

        "target_frames":
            total_target,

        "already_done":
            frames_already_done,

        "processed_now":
            processed,

        "aligned_frames":
            total_aligned,

        "skipped_frames":
            skipped,

        "failed_frames":
            failed,

        "fps":
            fps,

        "frame_count":
            frame_count,

        "elapsed_seconds":
            elapsed,

        "processing_speed":
            (
                processed / elapsed
                if elapsed > 0
                else 0
            ),

        "output_dir":
            str(
                paths[
                    "output_dir"
                ]
            ),

        "aligned_face_dir":
            str(
                paths[
                    "aligned_face"
                ]
            ),

        "landmarks_dir":
            str(
                paths[
                    "landmarks"
                ]
            ),

        "metadata":
            str(
                paths[
                    "alignment_metadata"
                ]
            ),

    }


    print()
    print("=" * 70)
    print(
        f"PHASE 3 COMPLETED : {video_id}"
    )
    print("=" * 70)

    print(
        f"Phase 2 rows   : "
        f"{len(detection_df)}"
    )

    print(
        f"Target frames  : "
        f"{total_target}"
    )

    print(
        f"Already done   : "
        f"{frames_already_done}"
    )

    print(
        f"Processed now  : "
        f"{processed}"
    )

    print(
        f"Aligned total  : "
        f"{total_aligned}"
    )

    print(
        f"Skipped        : "
        f"{skipped}"
    )

    print(
        f"Failed         : "
        f"{failed}"
    )

    print(
        f"Elapsed        : "
        f"{elapsed / 60:.2f} min"
    )

    print(
        f"Speed          : "
        f"{processed / elapsed:.2f} frame/s"
        if elapsed > 0
        else "Speed          : 0"
    )

    print("=" * 70)


    return stats

In [23]:
# ============================================================
# PHASE 3 — CELL 13
# FAST PRODUCTION BATCH RUNNER
# ============================================================

print()
print("=" * 70)
print("PHASE 3 — FAST PRODUCTION BATCH")
print("=" * 70)


# ============================================================
# Discover videos
# ============================================================

if not RAW_VIDEO_DIR.exists():

    raise FileNotFoundError(
        f"Raw video directory not found:\n"
        f"{RAW_VIDEO_DIR}"
    )


VIDEO_FILES = sorted(

    [

        p.resolve()

        for p in RAW_VIDEO_DIR.iterdir()

        if (
            p.is_file()
            and
            p.suffix.lower()
            in SUPPORTED_VIDEO_EXTENSIONS
        )

    ],

    key=lambda p:
        p.name.lower()

)


if not VIDEO_FILES:

    raise RuntimeError(
        "No supported videos found."
    )


print(
    f"Videos: {len(VIDEO_FILES)}"
)


# ============================================================
# Batch
# ============================================================

BATCH_RESULTS = []


batch_start = time.time()


for index, video_path in enumerate(
    VIDEO_FILES,
    start=1
):

    video_id = get_video_id(
        video_path
    )

    print()
    print(
        f"[{index}/{len(VIDEO_FILES)}] "
        f"{video_path.name}"
    )

    try:

        stats = process_video(
            video_path
        )

        BATCH_RESULTS.append(
            stats
        )


    except Exception as e:

        error_text = (
            f"{type(e).__name__}: {e}"
        )

        print()
        print(
            f"FAILED: {video_id}"
        )

        print(
            f"Error: {error_text}"
        )

        traceback.print_exc()


        BATCH_RESULTS.append({

            "video_id":
                video_id,

            "status":
                "FAILED",

            "error":
                error_text,

        })


    finally:

        gc.collect()

        try:

            cv2.setNumThreads(0)

        except Exception:

            pass


# ============================================================
# Summary
# ============================================================

batch_elapsed = (
    time.time()
    -
    batch_start
)


print()
print("=" * 70)
print("PHASE 3 — FAST BATCH SUMMARY")
print("=" * 70)


summary_rows = []


for result in BATCH_RESULTS:

    summary_rows.append({

        "video_id":
            result.get(
                "video_id"
            ),

        "status":
            result.get(
                "status"
            ),

        "target":
            result.get(
                "target_frames",
                0
            ),

        "already_done":
            result.get(
                "already_done",
                0
            ),

        "processed_now":
            result.get(
                "processed_now",
                0
            ),

        "aligned":
            result.get(
                "aligned_frames",
                0
            ),

        "skipped":
            result.get(
                "skipped_frames",
                0
            ),

        "failed":
            result.get(
                "failed_frames",
                0
            ),

        "minutes":
            round(
                result.get(
                    "elapsed_seconds",
                    0
                ) / 60,
                2
            ),

        "speed":
            round(
                result.get(
                    "processing_speed",
                    0
                ),
                2
            ),

    })


SUMMARY_DF = pd.DataFrame(
    summary_rows
)


if not SUMMARY_DF.empty:

    print(
        SUMMARY_DF.to_string(
            index=False
        )
    )


print()
print(
    f"Total batch time: "
    f"{batch_elapsed / 60:.2f} min"
)

print("=" * 70)


PHASE 3 — FAST PRODUCTION BATCH
Videos: 3

[1/3] video001.webm

PHASE 3 FAST PROCESSING : video001
Input     : C:\LipReadingSSL\dataset\raw_videos\video001.webm
Output    : C:\LipReadingSSL\output\video001
Aligned   : C:\LipReadingSSL\output\video001\aligned_face
Landmarks : C:\LipReadingSSL\output\video001\landmarks
Metadata  : C:\LipReadingSSL\output\video001\alignment_metadata.csv

Phase 2 rows : 43725
Target frames: 42267

Already done : 38768
Need process : 3499

FPS         : 25.0
Video frames: 43726


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Progress: 500/3499 | frame=41110 | 0.47 proc/s
Progress: 1000/3499 | frame=41610 | 0.79 proc/s
Progress: 1500/3499 | frame=42484 | 1.01 proc/s
Progress: 2000/3499 | frame=42985 | 1.19 proc/s

PHASE 3 COMPLETED : video001
Phase 2 rows   : 43725
Target frames  : 42267
Already done   : 38768
Processed now  : 2290
Aligned total  : 41058
Skipped        : 1209
Failed         : 0
Elapsed        : 29.98 min
Speed          : 1.27 frame/s

[2/3] video002.webm

PHASE 3 FAST PROCESSING : video002
Input     : C:\LipReadingSSL\dataset\raw_videos\video002.webm
Output    : C:\LipReadingSSL\output\video002
Aligned   : C:\LipReadingSSL\output\video002\aligned_face
Landmarks : C:\LipReadingSSL\output\video002\landmarks
Metadata  : C:\LipReadingSSL\output\video002\alignment_metadata.csv

Phase 2 rows : 39500
Target frames: 35613

Already done : 0
Need process : 35613

FPS         : 25.0
Video frames: 39501


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Progress: 500/35613 | frame=500 | 2.34 proc/s
Progress: 1000/35613 | frame=1229 | 2.40 proc/s
Progress: 1500/35613 | frame=1735 | 2.47 proc/s
Progress: 2000/35613 | frame=2371 | 2.49 proc/s
Progress: 2500/35613 | frame=2872 | 2.52 proc/s
Progress: 3000/35613 | frame=3376 | 2.53 proc/s
Progress: 3500/35613 | frame=4077 | 2.54 proc/s
Progress: 4000/35613 | frame=4578 | 2.54 proc/s
Progress: 4500/35613 | frame=5162 | 2.55 proc/s
Progress: 5000/35613 | frame=5809 | 2.54 proc/s
Progress: 5500/35613 | frame=6311 | 2.55 proc/s
Progress: 6000/35613 | frame=6811 | 2.55 proc/s
Progress: 6500/35613 | frame=7313 | 2.56 proc/s
Progress: 7000/35613 | frame=7815 | 2.56 proc/s
Progress: 7500/35613 | frame=8317 | 2.56 proc/s
Progress: 8000/35613 | frame=8819 | 2.55 proc/s
Progress: 8500/35613 | frame=9455 | 2.51 proc/s
Progress: 9000/35613 | frame=9957 | 2.49 proc/s
Progress: 9500/35613 | frame=10758 | 2.44 proc/s
Progress: 10000/35613 | frame=11260 | 2.42 proc/s
Progress: 10500/35613 | frame=11832 | 2

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Progress: 500/30422 | frame=509 | 2.63 proc/s
Progress: 1000/30422 | frame=1099 | 2.67 proc/s
Progress: 1500/30422 | frame=1809 | 2.69 proc/s
Progress: 2000/30422 | frame=3172 | 2.60 proc/s
Progress: 2500/30422 | frame=3912 | 2.63 proc/s
Progress: 3000/30422 | frame=4417 | 2.61 proc/s
Progress: 3500/30422 | frame=4950 | 2.56 proc/s
Progress: 4000/30422 | frame=5450 | 2.51 proc/s
Progress: 4500/30422 | frame=5956 | 2.48 proc/s
Progress: 5000/30422 | frame=6456 | 2.46 proc/s
Progress: 5500/30422 | frame=7721 | 2.44 proc/s
Progress: 6000/30422 | frame=8361 | 2.43 proc/s
Progress: 6500/30422 | frame=8911 | 2.43 proc/s
Progress: 7000/30422 | frame=9411 | 2.45 proc/s
Progress: 7500/30422 | frame=9967 | 2.47 proc/s
Progress: 8000/30422 | frame=10467 | 2.48 proc/s
Progress: 8500/30422 | frame=10967 | 2.50 proc/s
Progress: 9000/30422 | frame=11834 | 2.51 proc/s
Progress: 9500/30422 | frame=12385 | 2.52 proc/s
Progress: 10000/30422 | frame=12885 | 2.53 proc/s
Progress: 10500/30422 | frame=13638 

In [4]:
# ============================================================
# PHASE 4 — CLEANUP video002
# ลบเฉพาะ aligned face ที่ไม่อยู่ใน metadata
# ============================================================

from pathlib import Path
import pandas as pd
import re

VIDEO_ID = "video002"

video_dir = Path(r"C:\LipReadingSSL\output") / VIDEO_ID

aligned_dir = (
    video_dir / "aligned_face"
)

metadata_path = (
    video_dir / "alignment_metadata.csv"
)

# ============================================================
# Check paths
# ============================================================

if not aligned_dir.exists():
    raise FileNotFoundError(
        f"Aligned face directory not found:\n{aligned_dir}"
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata file not found:\n{metadata_path}"
    )

# ============================================================
# Load metadata
# ============================================================

df = pd.read_csv(
    metadata_path,
    low_memory=False
)

if "frame_index" not in df.columns:
    raise KeyError(
        "Column 'frame_index' not found in metadata."
    )

metadata_frames = set(
    pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )
    .dropna()
    .astype(int)
    .tolist()
)

# ============================================================
# Scan aligned files
# ============================================================

all_files = [
    p
    for p in aligned_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() == ".jpg"
]

# สำคัญ:
# ต้องใช้ \.jpg ไม่ใช่ \\.jpg
pattern = re.compile(
    r"^frame_(\d+)\.jpg$",
    re.IGNORECASE
)

remove_files = []
keep_files = []

for file_path in all_files:

    match = pattern.match(
        file_path.name
    )

    # --------------------------------------------------------
    # New format:
    # frame_000000.jpg
    # --------------------------------------------------------

    if match:

        frame_index = int(
            match.group(1)
        )

        if frame_index in metadata_frames:

            keep_files.append(
                file_path
            )

        else:

            remove_files.append(
                file_path
            )

    # --------------------------------------------------------
    # Old format:
    # 000000.jpg
    # 000000_0.jpg
    # etc.
    # --------------------------------------------------------

    else:

        remove_files.append(
            file_path
        )

# ============================================================
# Confirm
# ============================================================

print("=" * 70)
print(f"CLEANUP : {VIDEO_ID}")
print("=" * 70)

print(
    f"Metadata frames : "
    f"{len(metadata_frames)}"
)

print(
    f"Files before    : "
    f"{len(all_files)}"
)

print(
    f"Files to keep   : "
    f"{len(keep_files)}"
)

print(
    f"Files to delete : "
    f"{len(remove_files)}"
)

# ============================================================
# Safety check
# ============================================================

if len(keep_files) != len(metadata_frames):

    print()
    print("WARNING!")
    print(
        "Keep count does not match metadata count."
    )
    print(
        "No files will be deleted."
    )

    raise RuntimeError(
        "Cleanup aborted because KEEP count "
        "does not match metadata frame count."
    )

# ============================================================
# Delete
# ============================================================

deleted = 0

for file_path in remove_files:

    try:

        file_path.unlink()

        deleted += 1

    except Exception as e:

        print(
            f"DELETE FAILED: "
            f"{file_path.name}"
        )

        print(e)

# ============================================================
# Result
# ============================================================

remaining_files = [
    p
    for p in aligned_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() == ".jpg"
]

print()
print("=" * 70)
print("CLEANUP COMPLETED")
print("=" * 70)

print(
    f"Deleted   : {deleted}"
)

print(
    f"Remaining : {len(remaining_files)}"
)

print(
    f"Expected  : {len(metadata_frames)}"
)

print("=" * 70)

CLEANUP : video002
Metadata frames : 34794
Files before    : 73701
Files to keep   : 34794
Files to delete : 38907

CLEANUP COMPLETED
Deleted   : 38907
Remaining : 34794
Expected  : 34794
